# Day 24 — Lecture 10: Lasso Regression (Implementation)

> Source: Apna College Prime AI/ML • Day 24 Lecture 10 • Video Duration: `10:16`
> Core Objective: Scikit-learn me `Lasso` regression implement karna aur multiple `alpha` values loop karke best MSE dhoondhna.
> Key Parameters: `alpha=0.5`, `alphas = [0.001, 0.1, 1, 2, 5, 10, 20, 30, 40, 50, 100]`
> Dataset Used: `insurance.csv`
> Core Concept: Scikit-learn me math formula ka $\lambda$ parameter `alpha` kehlata hai — isey Gradient Descent ke learning rate se confuse nahi karna hai

## 🛠️ Step 0: Setup & Imports

Chalo `lasso_regression.ipynb` ke exact imports load karte hain (`Lasso`, metrics, `pandas`, `train_test_split`, `seaborn`).

In [ ]:
# 1. Exact libraries import karo
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
print("Imports ready!")

## 📍 Step 1: Data Load & Feature Engineering Pipeline

Wahi optimized `insurance.csv` features prepare karte hain (`drop_first=True`, `sex`/`smoker` map, `age_smoker`, `bmi_smoker`).

In [ ]:
# 1. Insurance dataset load karo
insurance_data = pd.read_csv("insurance.csv")

# 2. Encoding aur interaction features banao
X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]
X = pd.get_dummies(X, columns=["region"], drop_first=True, dtype=int)
X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})
X["age_smoker"] = X["age"] * X["smoker"]
X["bmi_smoker"] = X["bmi"] * X["smoker"]

# 3. 80-20 train-test split karo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## ⚠️ Step 2: Single `Lasso(alpha=0.5)` Fit & Parameter Note

Yaad rakhna:
- **Math $\lambda$ = sklearn `alpha`:** Non-negative float (`0` se `infinity`).
- **`alpha = 0` Warning:** Agar `alpha=0` rakho toh L1 penalty khatam hokar ye plain `LinearRegression` ban jata hai (sklearn khud OLS use karne ki salah deta hai).

In [ ]:
# 1. Random alpha=0.5 ke saath Lasso model create aur fit karo
lasso_model = Lasso(alpha=0.5)
lasso_model.fit(X_train, y_train)

# 2. Test data predict karke Mean Squared Error nikalo
y_pred = lasso_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("MSE for alpha=0.5:", mse)

## 🔬 Step 3: Hit-and-Trial Loop Across Multiple `alphas`

Hume pehle se nahi pata ki best `alpha` kya hoga, isliye multiple values test karke dekhte hain ki kis `alpha` par test MSE sabse kam aata hai.

In [ ]:
# 1. Candidate alphas aur empty mses list define karo
alphas = [0.001, 0.1, 1, 2, 5, 10, 20, 30, 40, 50, 100]
mses = []

# 2. Har alpha par model train karke MSE record karo
for a in alphas:
    lasso_model = Lasso(alpha=a)
    lasso_model.fit(X_train, y_train)
    y_pred = lasso_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"MSE for alpha={a}: ", mse)
    mses.append(mse)

## 📈 Step 4: `alpha` vs `MSE` Line Plot (`sns.lineplot`)

Dekho graph me MSE chhote `alpha` (~`10`) ke aas-paas minimum hai aur `20–100` par tezi se badhta hai. Agle lecture me manual loop ki jagah **Cross Validation (`LassoCV`)** seekhenge!

In [ ]:
# 1. Alphas vs MSEs ka lineplot draw karo
sns.lineplot(x=alphas, y=mses, marker="o")
plt.xlabel("alpha (Regularization Strength)")
plt.ylabel("Mean Squared Error (MSE)")
plt.title("Lasso Regression: Alpha vs MSE")
plt.show()